# ATLAS R2 — Qwen3-30B FullFinetune on MI300X
## FullFinetune + DeepSpeed + Flash-Attention
**Status:** R2 Training (Ronda 2)

---

### GPU Setup Check

In [ ]:
import torch
import transformers
import deepspeed
import flash_attn

print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"DeepSpeed: {deepspeed.__version__}")
print(f"Flash-Attention: {flash_attn.__version__}")
print(f"")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"GPU Compute Capability: {torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}")

### 1. Load Dataset

In [ ]:
from datasets import load_dataset
import json

# Load JSONL dataset
dataset_path = "/data/atlas_training_dataset.jsonl"

dataset = load_dataset(
    "json",
    data_files=dataset_path,
    split="train"
)

print(f"Dataset loaded: {len(dataset)} examples")
print(f"\nFirst example:")
print(json.dumps(dataset[0], indent=2)[:500])

### 2. Train/Eval Split

In [ ]:
# Split 90/10
split_data = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_data['train']
eval_dataset = split_data['test']

print(f"Train set: {len(train_dataset)} examples")
print(f"Eval set: {len(eval_dataset)} examples")
print(f"Total: {len(train_dataset) + len(eval_dataset)} examples")

### 3. Load Model & Tokenizer

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Rafaelcedav/atlas-core-30b-q8"

print(f"Loading model: {model_id}")
print("This may take 2-3 minutes...\n")

# Load with Flash-Attention
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print(f"✅ Model loaded")
print(f"   Params: {model.num_parameters():,}")
print(f"   Dtype: {next(model.parameters()).dtype}")
print(f"   Vocab size: {tokenizer.vocab_size}")

### 4. Tokenization

In [ ]:
def preprocess_function(examples, tokenizer, max_seq_length=2048):
    """Convert ChatML messages to tokens."""
    
    texts = []
    for messages in examples['messages']:
        text = ""
        for msg in messages:
            role = msg.get('role', '')
            content = msg.get('content', '')
            if role == 'system':
                text += f"<system>{content}</system>"
            elif role == 'user':
                text += f"<user>{content}</user>"
            elif role == 'assistant':
                text += f"<assistant>{content}</assistant>"
        texts.append(text)
    
    tokenized = tokenizer(
        texts,
        max_length=max_seq_length,
        truncation=True,
        padding="max_length",
    )
    
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

print("Tokenizing datasets...")
print("This may take 5-10 minutes...\n")

train_dataset_tokenized = train_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=train_dataset.column_names,
    num_proc=4,
    desc="Tokenizing train"
)

eval_dataset_tokenized = eval_dataset.map(
    lambda x: preprocess_function(x, tokenizer),
    batched=True,
    remove_columns=eval_dataset.column_names,
    num_proc=4,
    desc="Tokenizing eval"
)

print(f"✅ Tokenization complete")
print(f"   Train: {len(train_dataset_tokenized)} examples")
print(f"   Eval: {len(eval_dataset_tokenized)} examples")

### 5. Training Configuration

In [ ]:
from transformers import TrainingArguments
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"./outputs/r2_qwen3_30b_{timestamp}"

training_args = TrainingArguments(
    output_dir=output_dir,
    
    # Learning
    learning_rate=5e-5,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    warmup_steps=500,
    
    # Optimization
    optim="adamw_8bit",
    max_grad_norm=1.0,
    weight_decay=0.01,
    
    # Evaluation
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    
    # Logging
    logging_steps=10,
    log_level="info",
    report_to=["tensorboard"],
    
    # Hardware
    bf16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    dataloader_num_workers=4,
    
    seed=42,
    push_to_hub=False,
)

print(f"Training configuration:")
print(f"  Output dir: {output_dir}")
print(f"  Batch size (effective): {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Mixed precision: bfloat16")
print(f"  Gradient checkpointing: {training_args.gradient_checkpointing}")

### 6. Create Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_tokenized,
    eval_dataset=eval_dataset_tokenized,
    tokenizer=tokenizer,
)

print("✅ Trainer created and ready")
print(f"   Device: {trainer.args.device}")
print(f"   Distributed: {trainer.args.ddp_find_unused_parameters}")

### 7. START TRAINING 🚀

In [ ]:
print("╔════════════════════════════════════════════════════════╗")
print("║         ATLAS R2 TRAINING START                        ║")
print("║    FullFinetune + DeepSpeed + Flash-Attention         ║")
print("║    Qwen3-30B on AMD MI300X                            ║")
print("╚════════════════════════════════════════════════════════╝")
print("")

train_result = trainer.train()

print("")
print("╔════════════════════════════════════════════════════════╗")
print("║         ✅ TRAINING COMPLETE                           ║")
print("╚════════════════════════════════════════════════════════╝")
print(f"Final training loss: {train_result.training_loss:.4f}")

### 8. Evaluation

In [ ]:
print("Running final evaluation...")

eval_result = trainer.evaluate()

print(f"✅ Evaluation complete")
print(f"   Eval loss: {eval_result['eval_loss']:.4f}")
print(f"   Perplexity: {eval_result.get('eval_perplexity', 'N/A')}")

### 9. Save Model

In [ ]:
print(f"Saving model to {training_args.output_dir}...")

model.save_pretrained(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)

print(f"✅ Model saved")
print(f"   Path: {training_args.output_dir}")
print(f"   Params: {model.num_parameters():,}")

### 10. Test Inference

In [ ]:
from transformers import pipeline

# Load finetuned model
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# Test prompt
test_prompt = "<system>Eres ATLAS, auditor forense especializado en regulaciones financieras.</system><user>¿Cuál es el umbral de reporte del CSD bajo el Art. 17-H CFF?</user>"

output = pipe(
    test_prompt,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
)

print(f"Test prompt:")
print(test_prompt)
print(f"\nModel response:")
print(output[0]['generated_text'][len(test_prompt):])